# Managing Session

Strategy: Use Unique Thread IDs Per User/Session

The key is to create unique thread_id values that combine user and session information.

In [ ]:
# !pip3 install --upgrade certifi
# !pip3 install --trusted-host pypi.org --trusted-host pypi.python.org --trusted-host files.pythonhosted.org certifi
!pip3 install --upgrade pip

In [ ]:
!pip3 install langgraph-checkpoint-sqlite

## Setup: Import Dependencies

In [ ]:
from typing_extensions import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
import uuid
from datetime import datetime, timedelta
from pprint import pprint
import sqlite3
from dataclasses import dataclass
from typing import Optional, List

## 1. Hierarchical Threading (User + Workspace + Session)

This pattern allows organizing conversations by user, workspace, and session for complex multi-tenant applications.

In [ ]:
def get_hierarchical_thread_id(
    user_id: str,
    workspace_id: Optional[str] = None,
    session_id: Optional[str] = None
) -> str:
    """Generate hierarchical thread ID for multi-level isolation"""
    parts = [f"user_{user_id}"]

    if workspace_id:
        parts.append(f"workspace_{workspace_id}")

    if session_id:
        parts.append(f"session_{session_id}")

    return "_".join(parts)

# Examples of hierarchical thread IDs
print("Hierarchical Thread ID Examples:")
print("=" * 80)

# User only (all conversations for this user)
thread1 = get_hierarchical_thread_id("alice")
print(f"User only: {thread1}")

# User + Workspace (user's conversations in specific workspace)
thread2 = get_hierarchical_thread_id("alice", workspace_id="workspace_marketing")
print(f"User + Workspace: {thread2}")

# User + Workspace + Session (specific conversation)
thread3 = get_hierarchical_thread_id("alice", workspace_id="workspace_marketing", session_id="session_001")
print(f"User + Workspace + Session: {thread3}")

# Different user, same workspace
thread4 = get_hierarchical_thread_id("bob", workspace_id="workspace_marketing", session_id="session_002")
print(f"Different user: {thread4}")

In [ ]:
# Create a simple chatbot for testing
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str
    workspace_id: str
    session_id: str

def chatbot_node(state: ChatState):
    """Simple echo chatbot"""
    last_msg = state["messages"][-1]
    response = f"[{state['user_id']}@{state['workspace_id']}]: Echo - {last_msg}"
    return {"messages": [response]}

# Build graph
workflow = StateGraph(ChatState)
workflow.add_node("chat", chatbot_node)
workflow.add_edge(START, "chat")
workflow.add_edge("chat", END)

# Compile with checkpointer
checkpointer = MemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

print("\\n✅ Graph compiled with checkpointer")

In [ ]:
# Demonstrate hierarchical threading with multiple users
print("\\n" + "=" * 80)
print("HIERARCHICAL THREADING DEMO - Multiple Users & Workspaces")
print("=" * 80)

# Alice in Marketing workspace, Session 1
alice_thread = get_hierarchical_thread_id("alice", "marketing", "session_001")
config_alice = {"configurable": {"thread_id": alice_thread}}

result = graph.invoke({
    "messages": ["Hello from Alice!"],
    "user_id": "alice",
    "workspace_id": "marketing",
    "session_id": "session_001"
}, config_alice)
print(f"\\nAlice (Marketing/Session1): {result['messages'][-1]}")

# Bob in Engineering workspace, Session 1
bob_thread = get_hierarchical_thread_id("bob", "engineering", "session_001")
config_bob = {"configurable": {"thread_id": bob_thread}}

result = graph.invoke({
    "messages": ["Hello from Bob!"],
    "user_id": "bob",
    "workspace_id": "engineering",
    "session_id": "session_001"
}, config_bob)
print(f"Bob (Engineering/Session1): {result['messages'][-1]}")

# Alice in Marketing workspace, Session 2 (different conversation)
alice_thread2 = get_hierarchical_thread_id("alice", "marketing", "session_002")
config_alice2 = {"configurable": {"thread_id": alice_thread2}}

result = graph.invoke({
    "messages": ["This is Alice's second conversation"],
    "user_id": "alice",
    "workspace_id": "marketing",
    "session_id": "session_002"
}, config_alice2)
print(f"Alice (Marketing/Session2): {result['messages'][-1]}")

# Continue Alice's first conversation
result = graph.invoke({
    "messages": ["Continuing my first chat"],
    "user_id": "alice",
    "workspace_id": "marketing",
    "session_id": "session_001"
}, config_alice)
print(f"\\nAlice (Marketing/Session1) continued: {result['messages'][-1]}")
print(f"  Total messages in this thread: {len(result['messages'])}")

## 2. Managing Multiple Sessions: Database Approach

Track sessions in a database for production-grade session management.

In [ ]:
@dataclass
class Session:
    session_id: str
    user_id: str
    workspace_id: str
    thread_id: str
    created_at: datetime
    last_active: datetime
    message_count: int = 0

class SessionManager:
    """Database-backed session manager"""

    def __init__(self, db_path: str = ":memory:"):
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        self._create_tables()

    def _create_tables(self):
        """Create sessions table"""
        self.conn.execute('''
            CREATE TABLE IF NOT EXISTS sessions (
                session_id TEXT PRIMARY KEY,
                user_id TEXT NOT NULL,
                workspace_id TEXT NOT NULL,
                thread_id TEXT UNIQUE NOT NULL,
                created_at TIMESTAMP,
                last_active TIMESTAMP,
                message_count INTEGER DEFAULT 0
            )
        ''')

        # Create index for faster lookups
        self.conn.execute('''
            CREATE INDEX IF NOT EXISTS idx_user_workspace
            ON sessions(user_id, workspace_id)
        ''')
        self.conn.commit()

    def create_session(self, user_id: str, workspace_id: str) -> Session:
        """Create new session for user in workspace"""
        session_id = str(uuid.uuid4())[:8]  # Short ID for demo
        thread_id = get_hierarchical_thread_id(user_id, workspace_id, session_id)
        now = datetime.now()

        self.conn.execute('''
            INSERT INTO sessions
            (session_id, user_id, workspace_id, thread_id, created_at, last_active, message_count)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        ''', (session_id, user_id, workspace_id, thread_id, now, now, 0))
        self.conn.commit()

        return Session(session_id, user_id, workspace_id, thread_id, now, now, 0)

    def get_session(self, session_id: str) -> Optional[Session]:
        """Get session by ID"""
        cursor = self.conn.execute(
            'SELECT * FROM sessions WHERE session_id = ?',
            (session_id,)
        )
        row = cursor.fetchone()
        if row:
            return Session(
                row[0], row[1], row[2], row[3],
                datetime.fromisoformat(row[4]),
                datetime.fromisoformat(row[5]),
                row[6]
            )
        return None

    def get_user_sessions(self, user_id: str, workspace_id: Optional[str] = None) -> List[Session]:
        """Get all sessions for user (optionally filtered by workspace)"""
        if workspace_id:
            cursor = self.conn.execute('''
                SELECT * FROM sessions
                WHERE user_id = ? AND workspace_id = ?
                ORDER BY last_active DESC
            ''', (user_id, workspace_id))
        else:
            cursor = self.conn.execute('''
                SELECT * FROM sessions
                WHERE user_id = ?
                ORDER BY last_active DESC
            ''', (user_id,))

        return [
            Session(
                row[0], row[1], row[2], row[3],
                datetime.fromisoformat(row[4]),
                datetime.fromisoformat(row[5]),
                row[6]
            )
            for row in cursor.fetchall()
        ]

    def update_last_active(self, session_id: str, increment_messages: bool = False):
        """Update session last active time and optionally increment message count"""
        if increment_messages:
            self.conn.execute('''
                UPDATE sessions
                SET last_active = ?, message_count = message_count + 1
                WHERE session_id = ?
            ''', (datetime.now(), session_id))
        else:
            self.conn.execute('''
                UPDATE sessions
                SET last_active = ?
                WHERE session_id = ?
            ''', (datetime.now(), session_id))
        self.conn.commit()

    def delete_session(self, session_id: str):
        """Delete a session"""
        self.conn.execute('DELETE FROM sessions WHERE session_id = ?', (session_id,))
        self.conn.commit()

    def get_all_sessions(self) -> List[Session]:
        """Get all sessions"""
        cursor = self.conn.execute('SELECT * FROM sessions ORDER BY last_active DESC')
        return [
            Session(
                row[0], row[1], row[2], row[3],
                datetime.fromisoformat(row[4]),
                datetime.fromisoformat(row[5]),
                row[6]
            )
            for row in cursor.fetchall()
        ]

# Initialize session manager
session_mgr = SessionManager(":memory:")
print("✅ SessionManager initialized")

## 3. Complete Production Example: Multi-User Chat API

Simulate a production API with multiple users having multiple conversations.

In [ ]:
class ChatAPI:
    """Simulated production chat API"""

    def __init__(self, graph, session_manager: SessionManager):
        self.graph = graph
        self.session_mgr = session_manager

    def create_session(self, user_id: str, workspace_id: str) -> dict:
        """API: Create new chat session"""
        session = self.session_mgr.create_session(user_id, workspace_id)
        return {
            "session_id": session.session_id,
            "user_id": session.user_id,
            "workspace_id": session.workspace_id,
            "thread_id": session.thread_id,
            "created_at": session.created_at.isoformat()
        }

    def send_message(self, session_id: str, message: str) -> dict:
        """API: Send message in existing session"""
        # Get session
        session = self.session_mgr.get_session(session_id)
        if not session:
            return {"error": "Session not found"}

        # Prepare config with thread_id
        config = {"configurable": {"thread_id": session.thread_id}}

        # Invoke graph
        result = self.graph.invoke({
            "messages": [message],
            "user_id": session.user_id,
            "workspace_id": session.workspace_id,
            "session_id": session.session_id
        }, config)

        # Update session activity
        self.session_mgr.update_last_active(session_id, increment_messages=True)

        return {
            "session_id": session_id,
            "message": message,
            "response": result["messages"][-1],
            "total_messages": len(result["messages"])
        }

    def get_history(self, session_id: str) -> dict:
        """API: Get conversation history"""
        session = self.session_mgr.get_session(session_id)
        if not session:
            return {"error": "Session not found"}

        config = {"configurable": {"thread_id": session.thread_id}}
        state = self.graph.get_state(config)

        return {
            "session_id": session_id,
            "messages": state.values.get("messages", []),
            "message_count": len(state.values.get("messages", []))
        }

    def list_user_sessions(self, user_id: str, workspace_id: Optional[str] = None) -> dict:
        """API: List all sessions for user"""
        sessions = self.session_mgr.get_user_sessions(user_id, workspace_id)
        return {
            "user_id": user_id,
            "workspace_id": workspace_id,
            "session_count": len(sessions),
            "sessions": [
                {
                    "session_id": s.session_id,
                    "workspace_id": s.workspace_id,
                    "created_at": s.created_at.isoformat(),
                    "last_active": s.last_active.isoformat(),
                    "message_count": s.message_count
                }
                for s in sessions
            ]
        }

# Initialize API
api = ChatAPI(graph, session_mgr)
print("✅ ChatAPI initialized")

In [ ]:
# Simulate multiple users with multiple sessions
print("\n" + "=" * 80)
print("PRODUCTION SIMULATION - Multiple Users & Sessions")
print("=" * 80)

# User 1: Alice creates sessions in Marketing and Engineering
print("\n📱 Alice creates sessions...")
alice_marketing_session = api.create_session("alice", "marketing")
alice_eng_session = api.create_session("alice", "engineering")
print(f"  Marketing session: {alice_marketing_session['session_id']}")
print(f"  Engineering session: {alice_eng_session['session_id']}")

# User 2: Bob creates sessions
print("\n📱 Bob creates sessions...")
bob_marketing_session = api.create_session("bob", "marketing")
bob_sales_session = api.create_session("bob", "sales")
print(f"  Marketing session: {bob_marketing_session['session_id']}")
print(f"  Sales session: {bob_sales_session['session_id']}")

# User 3: Charlie creates one session
print("\n📱 Charlie creates session...")
charlie_eng_session = api.create_session("charlie", "engineering")
print(f"  Engineering session: {charlie_eng_session['session_id']}")

In [ ]:
# Simulate conversations in parallel
print("\n" + "=" * 80)
print("CONCURRENT CONVERSATIONS")
print("=" * 80)

# Alice in Marketing
print("\n💬 Alice (Marketing):")
resp = api.send_message(alice_marketing_session['session_id'], "Need campaign ideas")
print(f"  User: {resp['message']}")
print(f"  Bot: {resp['response']}")

# Bob in Marketing (different conversation, same workspace)
print("\n💬 Bob (Marketing):")
resp = api.send_message(bob_marketing_session['session_id'], "What's our budget?")
print(f"  User: {resp['message']}")
print(f"  Bot: {resp['response']}")

# Alice in Engineering (same user, different workspace)
print("\n💬 Alice (Engineering):")
resp = api.send_message(alice_eng_session['session_id'], "Deploy status?")
print(f"  User: {resp['message']}")
print(f"  Bot: {resp['response']}")

# Charlie in Engineering
print("\n💬 Charlie (Engineering):")
resp = api.send_message(charlie_eng_session['session_id'], "Running tests")
print(f"  User: {resp['message']}")
print(f"  Bot: {resp['response']}")

# Bob in Sales
print("\n💬 Bob (Sales):")
resp = api.send_message(bob_sales_session['session_id'], "Q4 targets?")
print(f"  User: {resp['message']}")
print(f"  Bot: {resp['response']}")

In [ ]:
# Continue conversations to show persistence
print("\n" + "=" * 80)
print("CONVERSATION CONTINUITY - Same sessions, different messages")
print("=" * 80)

# Alice continues in Marketing
print("\n💬 Alice (Marketing) - Message 2:")
resp = api.send_message(alice_marketing_session['session_id'], "Follow up on campaigns")
print(f"  Bot: {resp['response']}")
print(f"  Total messages in this session: {resp['total_messages']}")

# Bob continues in Marketing
print("\n💬 Bob (Marketing) - Message 2:")
resp = api.send_message(bob_marketing_session['session_id'], "Budget approved?")
print(f"  Bot: {resp['response']}")
print(f"  Total messages in this session: {resp['total_messages']}")

In [ ]:
# List sessions per user
print("\n" + "=" * 80)
print("SESSION OVERVIEW PER USER")
print("=" * 80)

# Alice's sessions
print("\n👤 Alice's sessions:")
alice_sessions = api.list_user_sessions("alice")
print(f"  Total sessions: {alice_sessions['session_count']}")
for session in alice_sessions['sessions']:
    print(f"    - {session['session_id']} ({session['workspace_id']}): {session['message_count']} messages")

# Bob's sessions
print("\n👤 Bob's sessions:")
bob_sessions = api.list_user_sessions("bob")
print(f"  Total sessions: {bob_sessions['session_count']}")
for session in bob_sessions['sessions']:
    print(f"    - {session['session_id']} ({session['workspace_id']}): {session['message_count']} messages")

# Charlie's sessions
print("\n👤 Charlie's sessions:")
charlie_sessions = api.list_user_sessions("charlie")
print(f"  Total sessions: {charlie_sessions['session_count']}")
for session in charlie_sessions['sessions']:
    print(f"    - {session['session_id']} ({session['workspace_id']}): {session['message_count']} messages")

In [ ]:
# Get conversation history for a specific session
print("\n" + "=" * 80)
print("CONVERSATION HISTORY")
print("=" * 80)

# Get Alice's Marketing conversation history
print(f"\n📜 Alice's Marketing conversation ({alice_marketing_session['session_id']}):")
history = api.get_history(alice_marketing_session['session_id'])
print(f"  Total messages: {history['message_count']}")
for i, msg in enumerate(history['messages'], 1):
    print(f"    {i}. {msg}")

## 4. Checkpoint Cleanup Strategy

Manage storage by removing old or inactive checkpoints.

In [ ]:
class CheckpointCleanupManager:
    """Manage checkpoint cleanup and archival"""

    def __init__(self, session_manager: SessionManager):
        self.session_mgr = session_manager

    def get_inactive_sessions(self, days: int = 30) -> List[Session]:
        """Find sessions inactive for N days"""
        cutoff = datetime.now() - timedelta(days=days)
        cursor = self.session_mgr.conn.execute('''
            SELECT * FROM sessions
            WHERE last_active < ?
            ORDER BY last_active ASC
        ''', (cutoff,))

        return [
            Session(
                row[0], row[1], row[2], row[3],
                datetime.fromisoformat(row[4]),
                datetime.fromisoformat(row[5]),
                row[6]
            )
            for row in cursor.fetchall()
        ]

    def archive_session(self, session_id: str):
        """Archive a session (mark for deletion)"""
        self.session_mgr.conn.execute('''
            UPDATE sessions
            SET archived = 1
            WHERE session_id = ?
        ''', (session_id,))
        # Note: Would need to add 'archived' column in production

    def delete_old_sessions(self, days: int = 90) -> int:
        """Delete sessions older than N days"""
        inactive_sessions = self.get_inactive_sessions(days)
        deleted_count = 0

        for session in inactive_sessions:
            self.session_mgr.delete_session(session.session_id)
            deleted_count += 1

        return deleted_count

    def get_session_stats(self) -> dict:
        """Get statistics about sessions"""
        cursor = self.session_mgr.conn.execute('''
            SELECT
                COUNT(*) as total_sessions,
                COUNT(DISTINCT user_id) as unique_users,
                COUNT(DISTINCT workspace_id) as unique_workspaces,
                SUM(message_count) as total_messages,
                AVG(message_count) as avg_messages_per_session
            FROM sessions
        ''')
        row = cursor.fetchone()

        return {
            "total_sessions": row[0],
            "unique_users": row[1],
            "unique_workspaces": row[2],
            "total_messages": row[3],
            "avg_messages_per_session": round(row[4], 2) if row[4] else 0
        }

    def cleanup_by_message_count(self, max_count: int = 0) -> int:
        """Delete sessions with no messages (abandoned)"""
        cursor = self.session_mgr.conn.execute('''
            SELECT session_id FROM sessions
            WHERE message_count <= ?
        ''', (max_count,))

        session_ids = [row[0] for row in cursor.fetchall()]

        for session_id in session_ids:
            self.session_mgr.delete_session(session_id)

        return len(session_ids)

# Initialize cleanup manager
cleanup_mgr = CheckpointCleanupManager(session_mgr)
print("✅ CheckpointCleanupManager initialized")

In [ ]:
# Get current session statistics
print("\n" + "=" * 80)
print("SESSION STATISTICS")
print("=" * 80)

stats = cleanup_mgr.get_session_stats()
print(f"\n📊 Current Stats:")
print(f"  Total sessions: {stats['total_sessions']}")
print(f"  Unique users: {stats['unique_users']}")
print(f"  Unique workspaces: {stats['unique_workspaces']}")
print(f"  Total messages: {stats['total_messages']}")
print(f"  Avg messages/session: {stats['avg_messages_per_session']}")

In [ ]:
# Demonstrate cleanup - find inactive sessions
print("\n" + "=" * 80)
print("CLEANUP DEMONSTRATION")
print("=" * 80)

# Check for sessions inactive for 0 days (for demo, this would be none)
# In production, you'd use 30, 60, 90 days
inactive = cleanup_mgr.get_inactive_sessions(days=0)
print(f"\n🧹 Sessions inactive for 0+ days: {len(inactive)}")

# Create a test session that we'll "abandon"
print("\n📱 Creating abandoned session for demo...")
abandoned_session = api.create_session("test_user", "test_workspace")
print(f"  Created session: {abandoned_session['session_id']}")

# Check stats before cleanup
stats_before = cleanup_mgr.get_session_stats()
print(f"\n📊 Before cleanup:")
print(f"  Total sessions: {stats_before['total_sessions']}")

# Cleanup sessions with 0 messages (abandoned sessions)
print("\n🧹 Cleaning up abandoned sessions (0 messages)...")
deleted = cleanup_mgr.cleanup_by_message_count(max_count=0)
print(f"  Deleted {deleted} abandoned session(s)")

# Check stats after cleanup
stats_after = cleanup_mgr.get_session_stats()
print(f"\n📊 After cleanup:")
print(f"  Total sessions: {stats_after['total_sessions']}")
print(f"  Cleaned up: {stats_before['total_sessions'] - stats_after['total_sessions']} session(s)")

## Summary: Production Best Practices

| **Aspect** | **Implementation** | **Purpose** |

|------------|-------------------|-------------|

| **Thread ID Format** | `user_{id}_workspace_{id}_session_{id}` | Hierarchical isolation |

| **Session Tracking** | SQLite database with SessionManager | Persistent session metadata |

| **Checkpointer** | `MemorySaver` (demo) or `SqliteSaver` (prod) | State persistence |

| **Cleanup Strategy** | Delete sessions inactive for 30-90 days | Manage storage costs |

| **API Design** | Stateless API with session-based routing | Scalable architecture |

### Key Takeaways

1. ✅ **One graph, multiple users** - Compile once, invoke many times

2. ✅ **Unique thread IDs** - Combine user + workspace + session for isolation

3. ✅ **Session database** - Track metadata separately from checkpoints

4. ✅ **Cleanup policy** - Regular cleanup of inactive/abandoned sessions

5. ✅ **Stats tracking** - Monitor session counts and message volumes